## This notebook will test using a tensor DNN to try to get similar performance to the paper

Will attempt to get a testing implementation working to try optimising parameters.

In [ ]:
import os
import time
from collections.abc import Sized, Iterable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

In [ ]:

csv_path = "./HIGGS.csv"
cache_npy = "./higgs_cache.npy"   # first run caches to .npy so reloads take seconds
n_rows = -1,                     # -1 = all 11M. Set e.g. 2_000_000 for quick dev runs

# --- feature set -----------------------------------------------------
# 28 columns total. Columns 1-21 are the "low-level" kinematic features,
# columns 22-28 are the 7 "high-level" derived features (m_jj ... m_wwbb).
# "all" reproduces the 0.885 result; "low" reproduces the 0.880 result and
# is the more impressive one -- the net rediscovers the high-level physics.
feature_set = "all"               # "all" | "low" | "high"

# --- architecture ----------------------------------------------------
# Paper's best: "five-layer" net = 4 hidden x 300 + output
hidden_layers = [300, 300, 300, 300, 300],
act_layer = nn.ReLU # nn.ReLU, nn.Tanh, or possibly nn.GELU
batchnorm = True                  # modern stabiliser the 2014 paper didn't have
dropout = 0.5                     # paper applied 50% dropout to the TOP hidden layer
dropout_all_layers = False        # False = dropout only on last hidden layer (paper-style)

# --- optimisation ----------------------------------------------------
epochs = 30                       # with 10M rows each epoch already sees a LOT of data
batch_size = 8192                 # big batches keep the GPU busy; paper used 100 (2011 GPU)
max_lr = 0.003                     # OneCycle peak LR for AdamW
weight_decay = 0.00001               # matches the paper's L2 coefficient
label_smoothing = 0.0             # try 0.01-0.05 if the net overfits

# --- engineering -----------------------------------------------------
amp = True                        # bfloat16 autocast (RTX 50-series supports it natively)
compile = True                    # torch.compile fuses kernels; big speedup, falls back if it fails
keep_data_on_gpu = True           # 10M x 28 float32 ~ 1.1 GB -> fits in 16 GB, avoids per-batch copies
early_stop_patience = 6           # stop if val AUC hasn't improved for this many epochs
seed = 1337

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")